# Reward and Architecture Study

This notebook investigates different reward functions and neural network architectures using PPO. It retrains six variants:

- `ppo_reward_very_simple`
- `ppo_reward_simple`
- `ppo_overtake_aggressive`
- `ppo_overtake_arch_128x3`
- `ppo_overtake_arch_256x2`
- `ppo_overtake_arch_256x3`

It saves training metrics, final models, evaluation results, and videos. Confidence intervals are calculated and a summary table is displayed.

In [ ]:
from pathlib import Path
import subprocess, sys, json, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Video

BASE_DIR = Path.cwd()
TRAIN_TIMESTEPS = 120_000
EVAL_EPISODES = 50
RUN_TRAINING = True
RUN_EVALUATION = True

EXPERIMENTS_TO_RUN = [
    'ppo_reward_very_simple',
    'ppo_reward_simple',
    'ppo_overtake_aggressive',
    'ppo_overtake_arch_128x3',
    'ppo_overtake_arch_256x2',
    'ppo_overtake_arch_256x3',
]

print(BASE_DIR)
print(EXPERIMENTS_TO_RUN)


## Training

Execute the training process for all variants.

In [ ]:
if RUN_TRAINING:
    for config_name in EXPERIMENTS_TO_RUN:
        cmd = [
            sys.executable,
            'train_sb3.py',
            '--algo', 'ppo',
            '--config', config_name,
            '--timesteps', str(TRAIN_TIMESTEPS),
        ]
        print(f'Executing: {" ".join(cmd)}')
        subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Training skipped as RUN_TRAINING is set to False.')

## Evaluation on 50 Episodes

Evaluate each model over 50 episodes for reliable metrics.

In [ ]:
if RUN_EVALUATION:
    for config_name in EXPERIMENTS_TO_RUN:
        model_path = BASE_DIR / 'runs' / f'ppo_{config_name}' / 'final_model.zip'
        if not model_path.exists():
            print(f'Model not found: {model_path}')
            continue
        cmd = [
            sys.executable,
            'evaluate_and_record.py',
            '--algo', 'ppo',
            '--config', config_name,
            '--model', str(model_path),
            '--episodes', str(EVAL_EPISODES),
            '--deterministic',
        ]
        print(f'Executing: {" ".join(cmd)}')
        subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Evaluation skipped as RUN_EVALUATION is set to False.')

## Loading Results

Load the results from training and evaluation.

In [ ]:
RUNS = {
    config_name: BASE_DIR / 'runs' / f'ppo_{config_name}'
    for config_name in EXPERIMENTS_TO_RUN
}

train_episode = {}
train_batch = {}
eval_episode = {}
eval_summary = {}

for name, run_dir in RUNS.items():
    ep_path = run_dir / 'train_episode_metrics.csv'
    batch_path = run_dir / 'train_batch_metrics.csv'
    eval_path = run_dir / 'evaluation_episodes.csv'
    summary_path = run_dir / 'evaluation_summary.json'

    if ep_path.exists():
        train_episode[name] = pd.read_csv(ep_path)
    if batch_path.exists():
        train_batch[name] = pd.read_csv(batch_path)
    if eval_path.exists():
        eval_episode[name] = pd.read_csv(eval_path)
    if summary_path.exists():
        with open(summary_path, 'r', encoding='utf-8') as f:
            eval_summary[name] = json.load(f)

print('Loaded train_episode data for:', list(train_episode.keys()))
print('Loaded train_batch data for:', list(train_batch.keys()))
print('Loaded eval_episode data for:', list(eval_episode.keys()))

## Training Curves per Batch/Update

Plot training metrics per batch/update.

In [ ]:
METRICS_TO_PLOT = [
    'episode_return',
    'episode_length',
    'overtakes',
    'overtaken_by_others',
    'lane_changes',
    'collision',
]

for metric in METRICS_TO_PLOT:
    plt.figure(figsize=(8, 4))
    plotted = False
    for name, df in train_batch.items():
        if metric in df.columns:
            x = df['update'] if 'update' in df.columns else np.arange(len(df)) + 1
            y = pd.to_numeric(df[metric], errors='coerce')
            if y.notna().any():
                plt.plot(x, y, label=name)
                plotted = True
    plt.title(f'Training batch metric: {metric}')
    plt.xlabel('update')
    plt.ylabel(metric)
    if plotted:
        plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Training Curves per Episode

Plot training metrics per episode with smoothing.

In [ ]:
for metric in METRICS_TO_PLOT:
    plt.figure(figsize=(8, 4))
    plotted = False
    for name, df in train_episode.items():
        if metric in df.columns:
            x = df['episode'] if 'episode' in df.columns else np.arange(len(df)) + 1
            y = pd.to_numeric(df[metric], errors='coerce')
            if y.notna().any():
                smooth = y.rolling(20, min_periods=1).mean()
                plt.plot(x, smooth, label=name)
                plotted = True
    plt.title(f'Training episode metric (rolling mean 20): {metric}')
    plt.xlabel('episode')
    plt.ylabel(metric)
    if plotted:
        plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Final Table with 95% Confidence Intervals

Display the final results table with confidence intervals.

In [ ]:
final_table = pd.DataFrame(rows)
cols = ['run'] + FINAL_METRICS
if not final_table.empty:
    display(final_table[cols])
else:
    print('No evaluation results available.')

## Raw Numeric View of Means

View the raw means of the metrics.

In [ ]:
numeric_cols = ['run'] + [f'{m}__mean' for m in FINAL_METRICS]
if not final_table.empty:
    display(final_table[numeric_cols].sort_values('overtakes__mean', ascending=False))


## Model Videos

Display videos of the trained models in action.

In [ ]:
for name, run_dir in RUNS.items():
    video_path = run_dir / 'videos' / 'eval_episode_1.mp4'
    if video_path.exists():
        print(name)
        display(Video(str(video_path), embed=True, width=700))
